# 03. Fixture'y i cykl życia testu

## Cele

Po tym notebooku potrafisz:

* przenieść wspólne przygotowanie testów do **fixture'a** (`TEST_F`, `SetUp`, `TearDown`),
* opisać cykl życia fixture'a: co jest tworzone na nowo dla każdego testu, a co jest współdzielone,
* pisać testy niezależne od siebie i od kolejności uruchamiania.

## Problem: powtarzane przygotowanie

Testy jednej klasy często zaczynają się tak samo — od zbudowania obiektu w znanym stanie:

```cpp
TEST(PlaylistTest, TotalDurationSumsTracks) {
    Playlist playlist;
    playlist.add("Intro", 90);
    playlist.add("Theme", 210);
    EXPECT_EQ(playlist.totalSeconds(), 300);
}

TEST(PlaylistTest, RemoveDeletesTrack) {
    Playlist playlist;
    playlist.add("Intro", 90);
    playlist.add("Theme", 210);
    playlist.remove("Intro");
    EXPECT_EQ(playlist.size(), 1u);
}
```

Powtórzona część *Arrange* zaciemnia to, co w każdym teście jest istotne, a zmiana danych startowych
wymaga poprawek w wielu miejscach. Rozwiązaniem jest **fixture** (w xUnit: *test fixture*) —
klasa, która przygotowuje wspólny stan dla grupy testów
([Primer — Test Fixtures](https://google.github.io/googletest/primer.html#same-data-multiple-tests)).

## `TEST_F`

1. Tworzymy klasę dziedziczącą po `::testing::Test`. Pola, z których korzystają testy, umieszczamy
   w sekcji `protected`.
2. Przygotowanie wpisujemy do konstruktora albo do metody `void SetUp() override`, a sprzątanie
   do destruktora albo `void TearDown() override`.
3. Testy definiujemy makrem `TEST_F` (*F* jak *fixture*), którego **pierwszym argumentem jest nazwa
   klasy fixture'a**. W treści testu pola fixture'a są dostępne bezpośrednio.

```cpp
#include <numeric>
#include <stdexcept>
#include <string>
#include <vector>

#include <gtest/gtest.h>

struct Track {
    std::string title;
    int seconds;
};

class Playlist {
public:
    void add(const std::string& title, int seconds) { tracks_.push_back({title, seconds}); }
    void remove(const std::string& title) {
        for (auto it = tracks_.begin(); it != tracks_.end(); ++it) {
            if (it->title == title) {
                tracks_.erase(it);
                return;
            }
        }
        throw std::out_of_range("no such track: " + title);
    }
    std::size_t size() const { return tracks_.size(); }
    int totalSeconds() const {
        return std::accumulate(tracks_.begin(), tracks_.end(), 0,
                               [](int sum, const Track& t) { return sum + t.seconds; });
    }

private:
    std::vector<Track> tracks_;
};

class PlaylistTest : public ::testing::Test {
protected:
    void SetUp() override {
        playlist.add("Intro", 90);
        playlist.add("Theme", 210);
    }

    Playlist playlist;
};

TEST_F(PlaylistTest, TotalDurationSumsTracks) {
    EXPECT_EQ(playlist.totalSeconds(), 300);
}

TEST_F(PlaylistTest, RemoveDeletesTrack) {
    playlist.remove("Intro");
    EXPECT_EQ(playlist.size(), 1u);
    EXPECT_EQ(playlist.totalSeconds(), 210);
}

TEST_F(PlaylistTest, RemoveMissingTrackThrows) {
    EXPECT_THROW(playlist.remove("Outro"), std::out_of_range);
}
```

Uwaga na literówkę: metoda musi nazywać się `SetUp` (duże „U”). Funkcja `Setup()` nie zostanie
wywołana, a kompilator nic nie powie — chyba że dopiszesz `override`, wtedy zgłosi błąd
([FAQ](https://google.github.io/googletest/faq.html#my-setup-function-is-not-called-why)).

W jednym pliku można mieszać `TEST` i `TEST_F`, ale nie w jednym zestawie: wszystkie testy zestawu muszą
korzystać z tej samej klasy fixture'a. `TEST(PlaylistTest, ...)` obok `TEST_F(PlaylistTest, ...)` się
skompiluje, ale przy uruchomieniu GoogleTest zgłosi błąd („mixing TEST_F and TEST in the same test suite”)
i taki test nie przejdzie.

## Cykl życia: nowy fixture dla każdego testu

Dla **każdego** testu zdefiniowanego przez `TEST_F` GoogleTest:

1. tworzy **nowy** obiekt fixture'a (konstruktor),
2. wywołuje `SetUp()`,
3. wykonuje treść testu,
4. wywołuje `TearDown()`,
5. niszczy obiekt (destruktor).

Dokumentacja mówi wprost: różne testy z tego samego zestawu mają **różne obiekty fixture'a**,
GoogleTest nie używa ponownie tego samego obiektu, więc zmiany, które jeden test wprowadzi
w polach fixture'a, nie wpływają na inne testy
([Primer](https://google.github.io/googletest/primer.html#same-data-multiple-tests)).
Każdy test zaczyna więc od świeżego, identycznego stanu — niezależnie od tego, co robiły testy przed nim.

Plik [`demos/lifecycle_test.cpp`](demos/lifecycle_test.cpp) wypisuje komunikat w każdym kroku.
Pierwszy test dodaje element do wektora, a drugi sprawdza, że go nie widzi.

In [ ]:
# Build and run the lifecycle demo (-V shows the program's output)
!cmake -S demos -B demos/build && cmake --build demos/build --config Debug --parallel && ctest --test-dir demos/build -C Debug -V

## Co jest współdzielone? Pola statyczne i `SetUpTestSuite`

Świeże są tylko zwykłe pola obiektu. **Pole statyczne** (`static`) należy do klasy, a nie do obiektu —
istnieje w jednym egzemplarzu przez cały czas działania programu, więc jego wartość „przechodzi”
z jednego testu do następnego. To zwykła reguła C++, nie specjalny mechanizm GoogleTest.

GoogleTest wykorzystuje to celowo do zasobów, które są kosztowne w tworzeniu (np. duży plik wczytany
raz dla wszystkich testów). Statyczne metody `SetUpTestSuite()` i `TearDownTestSuite()` są wywoływane
raz — przed pierwszym i po ostatnim teście zestawu
([Advanced — Sharing Resources Between Tests](https://google.github.io/googletest/advanced.html#sharing-resources-between-tests-in-the-same-test-suite)).
Dokumentacja dodaje dwa ostrzeżenia: kolejność testów nie jest zagwarantowana, a testy nie powinny
zmieniać współdzielonego stanu (albo muszą przywrócić go do stanu początkowego).

Istnieje też poziom całego programu: klasa `::testing::Environment`
([Global Set-Up and Tear-Down](https://google.github.io/googletest/advanced.html#global-set-up-and-tear-down)).

## Konstruktor czy `SetUp`?

FAQ GoogleTest zaleca zwykle **konstruktor i destruktor** — pozwalają np. oznaczyć pola jako `const`
([FAQ — constructor/destructor vs. SetUp/TearDown](https://google.github.io/googletest/faq.html#CtorVsSetUp)).
`SetUp`/`TearDown` są potrzebne, gdy:

* przygotowanie używa asercji `ASSERT_*` (w konstruktorze nie można ich użyć),
* przygotowanie wywołuje metody wirtualne (w konstruktorze nie działają polimorficznie),
* sprzątanie może rzucić wyjątek (destruktory są domyślnie `noexcept`, więc wyjątek z destruktora kończy program przez `std::terminate`).

Zauważ też, że `TestWithParam<T>` z notebooka 2 to również fixture — klasa testów sparametryzowanych
może mieć `SetUp` i pola jak każda inna.

## Niezależność testów

Testy, które po cichu zależą od siebie (np. jeden zostawia dane, z których korzysta drugi), przechodzą
albo nie w zależności od kolejności i zestawu uruchomionych testów. G. Meszaros nazywa je *Erratic Tests*,
a przyczynę — *Interacting Tests*; typowym źródłem jest współdzielony fixture
([Shared Fixture](http://xunitpatterns.com/Shared%20Fixture.html),
[Erratic Test](http://xunitpatterns.com/Erratic%20Test.html)).
To naruszenie zasady **I** z FIRST.

GoogleTest pomaga wykrywać takie zależności
([Advanced — Shuffling the Tests](https://google.github.io/googletest/advanced.html#shuffling-the-tests)):

* `--gtest_shuffle` uruchamia testy w losowej kolejności (ziarno losowania jest wypisywane,
  a `--gtest_random_seed=N` pozwala powtórzyć tę samą kolejność),
* `--gtest_repeat=N` powtarza testy N razy,
* `--gtest_filter=Wzorzec` uruchamia tylko wybrane testy — test, który przechodzi tylko razem z innymi,
  od razu się zdradzi.

Przykład: `./build/playlist_tests --gtest_shuffle --gtest_repeat=10`.

## Ćwiczenie 3 — `ShoppingCart` i fixture

Klasa `ShoppingCart` to koszyk w sklepie internetowym. Jej zachowanie opisują komentarze w pliku
[`src/shopping_cart.h`](src/shopping_cart.h) (kwoty w złotych, typ `double`):

| Metoda | Zachowanie |
|---|---|
| `add(name, unitPrice, quantity = 1)` | dodaje produkt; ponowne dodanie tego samego produktu zwiększa jego ilość |
| `remove(name)` | usuwa produkt (wszystkie sztuki); rzuca `std::out_of_range`, jeśli go nie ma |
| `itemCount()` | liczba sztuk wszystkich produktów |
| `total()` | suma `unitPrice × quantity` po rabacie |
| `applyDiscount(percent)` | ustawia rabat procentowy |
| `shippingCost()` | 0 od `kFreeShippingThreshold` (100,00 zł) wzwyż, poniżej `kShippingCost` (9,99 zł) |

W pliku <b>[tests/test_shopping_cart.cpp](tests/test_shopping_cart.cpp)</b> są cztery działające testy,
ale każdy zaczyna się od tych samych czterech linii przygotowania.

**Zadanie.**

1. Utwórz fixture `CartTest` (klasa dziedzicząca po `::testing::Test`) z polem `ShoppingCart cart`
   i przenieś wspólne przygotowanie (trzy produkty) do `SetUp()`.
2. Zamień cztery istniejące testy na `TEST_F(CartTest, ...)` — bez powtórzonego przygotowania.
3. Dopisz testy kolejnych zachowań:
   * pusty koszyk ma sumę 0 i zero sztuk (zastanów się: czy ten test powinien korzystać z fixture'a?),
   * usunięcie produktu, którego nie ma w koszyku, rzuca `std::out_of_range`,
   * ilość mnoży cenę jednostkową (np. 6 × 1,50 zł),
   * rabat procentowy obniża sumę (np. 10%),
   * dostawa jest płatna poniżej progu i darmowa **od** progu — sprawdź wartość dokładnie na progu
     (100,00 zł) i powyżej niego.

Sumy porównuj asercjami dla liczb zmiennoprzecinkowych (`EXPECT_DOUBLE_EQ` albo `EXPECT_NEAR`) — notebook 1.

### Zbuduj i uruchom testy

In [ ]:
# Configure, build and run the exercise tests
!cmake -S . -B build && cmake --build build --config Debug --parallel && ctest --test-dir build -C Debug --output-on-failure

Na początku przechodzą cztery istniejące testy. Po wykonaniu zadania wszystkie testy nadal przechodzą,
a jest ich około jedenastu, np. `100% tests passed, 0 tests failed out of 11`.

Sprawdź też, czy testy są naprawdę niezależne: uruchom je kilka razy w losowej kolejności
(GoogleTest umożliwia to opcją `--gtest_shuffle`; ścieżka do programu zależy od systemu, stąd `glob`):

In [ ]:
# Run the test program directly: random order, repeated 5 times
import glob, os, subprocess
exe = next(path for path in glob.glob("build/**/shopping_cart_tests*", recursive=True)
           if os.path.isfile(path) and os.path.basename(path) in ("shopping_cart_tests", "shopping_cart_tests.exe"))
result = subprocess.run([exe, "--gtest_shuffle", "--gtest_repeat=5", "--gtest_brief=1"], capture_output=True, text=True)
print(result.stdout[-1500:])

## Linki i materiały

* [Primer — Test Fixtures](https://google.github.io/googletest/primer.html#same-data-multiple-tests)
* [FAQ — konstruktor/destruktor czy SetUp/TearDown](https://google.github.io/googletest/faq.html#CtorVsSetUp)
* [Advanced — Sharing Resources Between Tests in the Same Test Suite](https://google.github.io/googletest/advanced.html#sharing-resources-between-tests-in-the-same-test-suite)
* [FAQ — dlaczego fixture'y zamiast zmiennych globalnych](https://google.github.io/googletest/faq.html#why-should-i-prefer-test-fixtures-over-global-variables)
* [G. Meszaros — Fresh Fixture](http://xunitpatterns.com/Fresh%20Fixture.html) i [Shared Fixture](http://xunitpatterns.com/Shared%20Fixture.html)

---

[← 02. Testy sparametryzowane](../02_parametrized/02_parametrized.ipynb) | [Strona główna](../_start.ipynb) | [04. Dublery testowe i gMock →](../04_mocks/04_mocks.ipynb)